<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">Exercises 4 to 7. Molecules are not points, and every check has a definition</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Module 3 (Molecules are not points) and module 4 (Every check has a definition). Four tasks, 50 minutes.</em>

In the last notebook a sample was a point in the plane and quality was one number.
Now a sample is a molecule: a set of atoms with elements, coordinates and implied
bonds, and it can fail in ways a point cannot. It can be disconnected. It can have a
carbon making five bonds. It can sit in a protein pocket at a distance no two atoms
physically occupy. Each of those failures has a checker, and each checker has a
definition, and the definition is where this notebook spends its time.

Four tasks. X4 asks what a saturation curve is a curve of. X5 hands you a
verification result and a measurement that cannot both be true, and asks you to work
out which of your assumptions is wrong. X6 computes the same quantity twice, two
defensible ways, and gets two different answers. X7 sorts four numbers by how much
weight they can carry.

X5 is the longest and it is the one to do properly. Set aside twenty minutes and
resist opening the solution; the exercise only works once.

<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Objectives</strong>
<ol style="margin:8px 0 0;padding-left:22px;font-size:15.5px;color:#241c1d;">
<li style="margin:6px 0;">Read a saturation curve and state which metric it saturated in</li>
<li style="margin:6px 0;">Reconcile a verification result against a measurement that contradicts it</li>
<li style="margin:6px 0;">Compute a similarity-derived quantity two defensible ways and report both</li>
<li style="margin:6px 0;">Rank measurements by reproducibility and say what would move each one</li>
</ol>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #6B6461;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#6B6461;">Runtime</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Everything in this notebook runs on a laptop CPU in well under a minute of compute. There is no download, no GPU, no API key and no network call; every number you produce comes from a file shipped inside this bundle.</span>
</div>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #B4690E;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#B4690E;">Caution</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Read every metrics CSV in this bundle with <code>gv.read_metrics_csv</code>. The files carry provenance as leading <code>#</code> lines, and <code>#</code> is also the SMILES symbol for a triple bond, so passing <code>comment='#'</code> to pandas silently truncates any row holding a nitrile: 16 cells go NaN in <code>boltz2_comparison.csv</code> and 26 in <code>novelty_diffsbdd.csv</code>, with no error raised.</span>
</div>

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #A20025;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#A20025;">Note</strong><br>
<span style="color:#3d3436;font-size:14.5px;">Each task ends with a self-check cell. It tells you whether your answer sits inside the tolerance for that task and, when it does not, which way to look. It never prints the answer. The worked solutions live in <code>solutions/</code>; open them after you have an answer of your own, not before.</span>
</div>

In [ ]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

In [ ]:
# Shared setup for all four tasks. RDKit and numpy only; no model, no download.
import numpy as np
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
print("rdkit", Chem.rdBase.rdkitVersion)

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X4&nbsp;&middot;&nbsp;introductory&nbsp;&middot;&nbsp;8 min&nbsp;&middot;&nbsp;M3&nbsp;&middot;&nbsp;O7, O8, O9</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Where does the curve stop paying you</h2>

FlowMol3 generates molecules in three dimensions by integrating a learned velocity
field, and like the toy sampler in X1 it has a step budget. The shipped table
records what a batch of twelve molecules looked like at five settings of that budget.

Report chemical validity at 10, 25 and 50 integration steps, and state the smallest
budget at which validity has saturated. Then look at the mean synthetic-accessibility
column over the same budgets, where a lower score means a chemist finds the molecule
easier to make, and answer one question: is 50 steps the right answer to "how many
steps do I need"? Justify it in two sentences.

One thing to carry into your answer. The batch size is twelve at every setting, so
every percentage in this table is a fraction with twelve on the bottom. A validity of
41.7 percent is five molecules out of twelve. Percentages on a denominator that small
need their denominator stated, every time.

In [ ]:
# X4. The FlowMol3 step-count table.
fm3 = gv.read_metrics_csv(DATA / "flowmol3_metrics.csv")
print(fm3.to_string(index=False))

# YOUR CODE HERE: read the three validity percentages off the table.
validity_10 = None
validity_25 = None
validity_50 = None

# The smallest budget at which validity has saturated (stops improving).
validity_saturates_at = None      # YOUR CODE HERE

# And the mean SA at 50 and at 250 steps.
sa_50 = None                      # YOUR CODE HERE
sa_250 = None                     # YOUR CODE HERE

In [ ]:
# X4, the judgement. Is 50 steps the right answer to "how many steps do I need"?
# Two sentences. Name what the answer depends on.
step_budget_answer = ""    # YOUR ANSWER HERE

gv.check_value("X4.val10", validity_10)
gv.check_value("X4.val25", validity_25)
gv.check_value("X4.val50", validity_50)
gv.check_value("X4.saturation", validity_saturates_at)
gv.check_value("X4.sa50", sa_50)
gv.check_value("X4.sa250", sa_250)
gv.check_written("X4.judgement", step_budget_answer,
                  "Validity saturates at one budget. Does the SA column?")
gv.checklist_summary("X4")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X5&nbsp;&middot;&nbsp;advanced&nbsp;&middot;&nbsp;20 min&nbsp;&middot;&nbsp;M4&nbsp;&middot;&nbsp;O10, O11</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">The check that could not have failed</h2>

PoseBusters is the standard physical-validity battery for a generated or docked
ligand pose. Given a molecule and a receptor it runs a fixed list of checks: eleven
about the molecule on its own (are the bond lengths plausible, is the ring flat when
it should be, does the molecule clash with itself) and nine about the molecule
against its surroundings (is it too far from the protein, does it overlap the
protein, does it overlap an organic cofactor, does it overlap a water).

The shipped table below is that battery run over twenty pocket-conditioned ligands
generated by DiffSBDD against the adenosine A2A receptor, structure 3RFM. The two
columns to notice are `minimum_distance_to_organic_cofactors` and
`volume_overlap_with_organic_cofactors`. Both read True for all twenty ligands.

Four parts.

1. Count the HETATM records in `data/3rfm_protein.pdb`. Report the count, and while
   you are there, count the ATOM records too.
2. For each of the twenty ligands in `data/diffsbdd_generated.sdf`, measure the
   smallest distance from any of its atoms to any atom of the co-crystallized
   caffeine in `data/3rfm_caffeine_ref.sdf`. Report the median and the largest of
   those twenty distances.
3. Reconcile parts one and two against the reported result, and state what the
   overall verdict would have been if the receptor file had retained its bound
   ligand.
4. Say which preparation is the right one, and why the answer is not obvious.

Part four has no key. It is a scientific judgement, and the grading rubric is whether
you can name a case where the opposite preparation is correct.

<div style="margin:0 0 16px;padding:2px 0 2px 18px;border-left:2px solid #6B6461;">
<strong style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:10.5px;letter-spacing:.16em;text-transform:uppercase;color:#6B6461;">Runtime</strong><br>
<span style="color:#3d3436;font-size:14.5px;">About ten seconds. RDKit, numpy and a text parse; no PoseBusters install required, because you are reading the battery's committed output rather than re-running it.</span>
</div>

In [ ]:
# X5, the shipped result. Twenty ligands, the full protein-aware battery.
pb = gv.read_metrics_csv(DATA / "posebusters_diffsbdd.csv")

INTER_CHECKS = ["protein-ligand_maximum_distance", "minimum_distance_to_protein",
                "minimum_distance_to_organic_cofactors",
                "minimum_distance_to_inorganic_cofactors", "minimum_distance_to_waters",
                "volume_overlap_with_protein", "volume_overlap_with_organic_cofactors",
                "volume_overlap_with_inorganic_cofactors", "volume_overlap_with_waters"]

print("the nine intermolecular checks, pass count out of", len(pb))
for c in INTER_CHECKS:
    print(f"  {c:<44s} {int(pb[c].sum())}/{len(pb)}")
print()
print("fully valid (all evaluated checks pass):", int(pb.pb_valid.sum()), "of", len(pb))

In [ ]:
# X5, part 1. What does the receptor file contain?
#
# gv.pdb_record_census(path) counts PDB lines by their record keyword, which is the
# first six columns of each line. A protein structure file usually carries ATOM
# records for the polymer and HETATM records for everything else: ligands, cofactors,
# ions, waters.
census = gv.pdb_record_census(DATA / "3rfm_protein.pdb")
print(census)

n_hetatm = None      # YOUR CODE HERE
n_atom = None        # YOUR CODE HERE

In [ ]:
# X5, part 2. How close is each generated ligand to the co-crystallized caffeine?
#
# gv.read_sdf(path) returns a list of RDKit molecules with 3D coordinates intact.
# mol.GetConformer().GetPositions() gives you an (n_atoms, 3) numpy array.
caffeine = gv.read_sdf(DATA / "3rfm_caffeine_ref.sdf")[0]
ligands = gv.read_sdf(DATA / "diffsbdd_generated.sdf")
caffeine_xyz = caffeine.GetConformer().GetPositions()
print(f"{len(ligands)} generated ligands | caffeine has {len(caffeine_xyz)} atoms")

closest_approach = []
for mol in ligands:
    # YOUR CODE HERE: compute the smallest distance between any atom of `mol` and
    # any atom of caffeine, and append it to closest_approach.
    pass

median_closest = float(np.median(closest_approach)) if closest_approach else None
max_closest = float(np.max(closest_approach)) if closest_approach else None
n_within_06 = sum(1 for d in closest_approach if d < 0.6)

print("median closest approach:", median_closest)
print("largest closest approach:", max_closest)
print("ligands with an atom within 0.6 angstroms of caffeine:", n_within_06,
      "of", len(ligands))

In [ ]:
# X5, part 3. Reconcile.
#
# You now have two facts that sit awkwardly together: the table says every ligand
# passes both organic-cofactor checks, and your own measurement puts every ligand
# within 0.6 angstroms of the caffeine that defined this pocket.
#
# What would the fully-valid count have been if the receptor file had retained its
# bound ligand?
valid_if_caffeine_retained = None     # YOUR CODE HERE (a count out of 20)

# And say why, in two or three sentences. What is a verification result a statement
# about?
reconciliation = ""                   # YOUR ANSWER HERE

In [ ]:
# X5, part 4. Which preparation is right?
#
# There is no key for this one. Answer in three or four sentences, and make sure your
# answer names a case where the OPPOSITE preparation would be the correct one.
which_preparation = ""     # YOUR ANSWER HERE

gv.check_value("X5.hetatm", n_hetatm)
gv.check_value("X5.atom", n_atom)
gv.check_value("X5.median_closest", median_closest)
gv.check_value("X5.max_closest", max_closest)
gv.check_value("X5.n_under_06", n_within_06)
gv.check_value("X5.pb_reported", int(pb.pb_valid.sum()))
gv.check_value("X5.pb_if_retained", valid_if_caffeine_retained)
gv.check_written("X5.reconciliation", reconciliation,
                  "What input did the verifier receive, and what could that input "
                  "have caused a check to fail on?")
gv.check_written("X5.preparation", which_preparation,
                  "Name a bound molecule you must NOT drop from a receptor, and say "
                  "what makes it different from this one.")
gv.checklist_summary("X5")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X6&nbsp;&middot;&nbsp;intermediate&nbsp;&middot;&nbsp;10 min&nbsp;&middot;&nbsp;M4&nbsp;&middot;&nbsp;O12</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Two correct answers to one question</h2>

Diversity is the standard one-number summary of a generated batch: one minus the mean
pairwise Tanimoto similarity, averaged over every unordered pair. It appears in
almost every generative-chemistry paper, usually as a single figure with no
qualification.

Compute it twice for the twenty ligands you already have in memory. Once with the
RDKit topological fingerprint at library defaults, and once with a Morgan fingerprint
at radius 2 and 2048 bits. Report both. Then decide which one you would put in a
paper, and write the sentence you would put beside it.

Two traps are graded. Hydrogen handling is the first: fingerprint these molecules
with explicit hydrogens attached and the RDKit value comes out 0.4777 instead, so
pin the convention with `Chem.RemoveHs` and say which convention you used. The second
is the denominator. The mean is over pairs, not over molecules: twenty molecules give
190 unordered pairs, and dividing by 20 instead produces a negative number, which is
not a diversity at all.

In [ ]:
# X6. Both fingerprints, one batch of molecules.
from rdkit import DataStructs
from rdkit.Chem import rdFingerprintGenerator


def diversity(mols, fingerprint):
    """1 minus the mean pairwise Tanimoto over all unordered pairs.

    `fingerprint` is 'rdkit' (topological, library defaults) or 'morgan'
    (circular, radius 2, 2048 bits). Return (diversity, n_pairs).
    """
    # YOUR CODE HERE. Pin the hydrogen convention, and remember what the mean is
    # taken over.
    return None, None


div_rdkit, n_pairs_rdkit = diversity(ligands, "rdkit")
div_morgan, n_pairs_morgan = diversity(ligands, "morgan")

print(f"RDKit topological (defaults):  {div_rdkit}   over {n_pairs_rdkit} pairs")
print(f"Morgan r=2, 2048 bits:         {div_morgan}   over {n_pairs_morgan} pairs")

In [ ]:
# X6. The sentence you would put in a paper beside whichever number you chose.
# It has to name the fingerprint, its parameters, and what the mean was taken over.
paper_sentence = ""       # YOUR ANSWER HERE

gv.check_value("X6.div_rdkit", div_rdkit)
gv.check_value("X6.div_morgan", div_morgan)
gv.check_value("X6.n_pairs", n_pairs_rdkit)
gv.check_written("X6.sentence", paper_sentence,
                  "A reader has to be able to reproduce your number from your "
                  "sentence alone.")
gv.checklist_summary("X6")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.2em;text-transform:uppercase;color:#A20025;margin:0 0 8px;">X7&nbsp;&middot;&nbsp;advanced&nbsp;&middot;&nbsp;12 min&nbsp;&middot;&nbsp;M4&nbsp;&middot;&nbsp;O13</div>
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Sort four numbers by how much you can trust them</h2>

Four verification numbers from this batch, all from files you have already read.

1. The DiffSBDD median UFF strain energy, from PoseCheck.
2. The PoseCheck clash count for the same batch.
3. The batch's pairwise diversity, which you computed in X6.
4. The pass rate on `minimum_distance_to_organic_cofactors`, which you met in X5.

For each, decide whether re-running the measurement on the same input would return
the same value, and say what would make it change. Then rank the four from most to
least trustworthy as a reported number, and defend the ranking.

The classifications to choose between are `reproduces` (same input, same value, every
time), `drifts` (same input, different value, because something inside the tool is
unseeded), `definition-dependent` (reproduces exactly, but a different defensible
definition gives a different number), and `vacuous` (the check could not have
returned anything else on this input).

Notice one asymmetry before you classify them: two of these four numbers come
out of the same tool and behave differently.

In [ ]:
# X7. The four numbers, with their sources.
pc = gv.read_metrics_csv(DATA / "posecheck_diffsbdd.csv")
lig_rows = pc[~pc.is_reference]
ref_row = pc[pc.is_reference]

strain_median = float(lig_rows.strain_energy_kcal_mol.median())
strain_reference = float(ref_row.strain_energy_kcal_mol.iloc[0])
clash_median = float(lig_rows.clash_count.median())
cofactor_pass_rate = float(pb["minimum_distance_to_organic_cofactors"].mean() * 100)

print(f"1. median UFF strain energy      {strain_median:8.1f} kcal/mol  "
      f"(caffeine reference {strain_reference:.2f}, a ratio of about "
      f"{strain_median / strain_reference:.0f})")
print(f"2. median clash count            {clash_median:8.1f}")
print(f"3. pairwise diversity            {div_rdkit if div_rdkit else float('nan'):8.4f}"
      "  (RDKit topological)")
print(f"4. cofactor minimum-distance     {cofactor_pass_rate:8.1f} percent passing")

Two pieces of evidence you are allowed to use, both taken verbatim from the notebooks
that produced these files.

The PoseCheck strain calculation compares a pose's energy against the energy of a
relaxed conformer ensemble, and the notebook's own comment records how that ensemble
is built: `AllChem.EmbedMolecule` with no `randomSeed` passed. The clash count is
computed from the fixed input geometry with a fixed distance tolerance and involves
no embedding step at all.

The diversity metadata file shipped alongside the batch records both fingerprint
definitions explicitly, along with the number of pairs and the preparation step. Open
it if you want to see what a definition looks like when somebody writes it down.

In [ ]:
# The definitions file for the diversity number, if you want to read it.
import json

meta = json.loads((DATA / "diversity_metadata.json").read_text())
print(json.dumps(meta["fingerprint_definitions"], indent=1))

In [ ]:
# X7. Classify each of the four. Use one of:
#     "reproduces" | "drifts" | "definition-dependent" | "vacuous"
strain_class = ""        # YOUR ANSWER HERE
clash_class = ""         # YOUR ANSWER HERE
diversity_class = ""     # YOUR ANSWER HERE
cofactor_class = ""      # YOUR ANSWER HERE

# Which of the four did you rank LAST, as the least trustworthy to report?
# Name it in a word or short phrase.
ranked_last = ""         # YOUR ANSWER HERE

# And the defence of your ranking, in three or four sentences. The order matters
# less than the reasons.
ranking_defence = ""     # YOUR ANSWER HERE

In [ ]:
# X7 self-check.
gv.check_text("X7.strain", strain_class)
gv.check_text("X7.clash", clash_class)
gv.check_text("X7.diversity", diversity_class)
gv.check_text("X7.cofactor", cofactor_class)
gv.check_text("X7.last_in_ranking", ranked_last)
gv.check_written("X7.defence", ranking_defence,
                  "Two of these four come from one tool and behave differently. "
                  "Does your defence say why?")
gv.checklist_summary("X7")

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Before you move on</h2>

In X5 you met a verification table and a distance measurement, both accurate,
that contradicted each other until you inspected the PDB file handed to the
verifier. Six of the nine intermolecular checks in that table read 100 percent
beside three protein checks that could fail, with nothing in the output
separating them.

PoseBusters ran the exact checks its input allowed. Every verification result in
the next notebook and in the papers you read afterward depends the same way on
how its input was prepared.

Next: `EX3_second_opinions_and_transfer.ipynb`.